# SQL Question 03 — JOIN Fan-Out and Incorrect Aggregation

## Problem
Expected claim amount: 300
Incorrect amount after JOIN: 400

## Objective
Understand one-to-many joins, data grain,
join amplification, and correct aggregation.


In [1]:

# Q3 Step 1: Create sample SQLite tables

import sqlite3
import pandas as pd
from pathlib import Path

# Locate our project folder
project_q3 = Path.home() / "question_03_join_fanout"

# Ensure the project directory exists
project_q3.mkdir(parents=True, exist_ok=True)

# Create a persistent SQLite database
db_path_q3 = project_q3 / "claims_join.db"
conn_q3 = sqlite3.connect(str(db_path_q3))

# Create tables without deleting existing data
conn_q3.execute("""
CREATE TABLE IF NOT EXISTS claims (
    claim_id INTEGER PRIMARY KEY,
    total_amount INTEGER NOT NULL
)
""")

conn_q3.execute("""
CREATE TABLE IF NOT EXISTS claim_lines (
    line_id INTEGER PRIMARY KEY,
    claim_id INTEGER NOT NULL,
    line_code TEXT NOT NULL
)
""")

# Insert sample claims only if their IDs are absent
conn_q3.executemany("""
INSERT OR IGNORE INTO claims (claim_id, total_amount)
VALUES (?, ?)
""", [
    (1, 100),
    (2, 200)
])

# Insert claim lines
conn_q3.executemany("""
INSERT OR IGNORE INTO claim_lines
(line_id, claim_id, line_code)
VALUES (?, ?, ?)
""", [
    (11, 1, "A"),
    (12, 1, "B"),
    (13, 2, "C")
])

conn_q3.commit()

print("CLAIMS TABLE:")
display(pd.read_sql_query(
    "SELECT * FROM claims ORDER BY claim_id",
    conn_q3
))

print("CLAIM LINES TABLE:")
display(pd.read_sql_query(
    "SELECT * FROM claim_lines ORDER BY line_id",
    conn_q3
))

print("Database:", db_path_q3)


CLAIMS TABLE:


,claim_id,total_amount
0,1,100
1,2,200


CLAIM LINES TABLE:


,line_id,claim_id,line_code
0,11,1,A
1,12,1,B
2,13,2,C


Database: /home/itv028683/question_03_join_fanout/claims_join.db


In [3]:

# Q3 Step 2: Reproduce incorrect total after INNER JOIN

incorrect_join_sql_q3 = """
SELECT
    SUM(c.total_amount) AS incorrect_total
FROM claims AS c
INNER JOIN claim_lines AS cl
    ON c.claim_id = cl.claim_id;
"""

incorrect_result_q3 = pd.read_sql_query(
    incorrect_join_sql_q3,
    conn_q3
)

display(incorrect_result_q3)

print(
    "Incorrect JOIN total:",
    incorrect_result_q3.loc[0, "incorrect_total"]
)


,incorrect_total
0,400


Incorrect JOIN total: 400


In [4]:

# Q3 Step 3: Inspect individual INNER JOIN rows

joined_rows_sql_q3 = """
SELECT
    c.claim_id,
    c.total_amount,
    cl.line_id,
    cl.line_code
FROM claims AS c
INNER JOIN claim_lines AS cl
    ON c.claim_id = cl.claim_id
ORDER BY c.claim_id, cl.line_id;
"""

joined_rows_df_q3 = pd.read_sql_query(
    joined_rows_sql_q3,
    conn_q3
)

display(joined_rows_df_q3)

print("Joined row count:", len(joined_rows_df_q3))
print(
    "Sum of repeated header amounts:",
    joined_rows_df_q3["total_amount"].sum()
)


,claim_id,total_amount,line_id,line_code
0,1,100,11,A
1,1,100,12,B
2,2,200,13,C


Joined row count: 3
Sum of repeated header amounts: 400


In [5]:

# Q3 Step 4: Identify JOIN fan-out per claim

fanout_sql_q3 = """
SELECT
    c.claim_id,
    COUNT(cl.line_id) AS matching_lines,
    c.total_amount
FROM claims AS c
LEFT JOIN claim_lines AS cl
    ON c.claim_id = cl.claim_id
GROUP BY
    c.claim_id,
    c.total_amount
ORDER BY c.claim_id;
"""

fanout_df_q3 = pd.read_sql_query(
    fanout_sql_q3,
    conn_q3
)

display(fanout_df_q3)

print(
    "Claims with multiple matching lines:",
    (fanout_df_q3["matching_lines"] > 1).sum()
)


,claim_id,matching_lines,total_amount
0,1,2,100
1,2,1,200


Claims with multiple matching lines: 1


In [6]:

# Q3 Step 5: Fix JOIN fan-out using EXISTS

correct_sql_q3 = """
SELECT
    SUM(c.total_amount) AS correct_total
FROM claims AS c
WHERE EXISTS (
    SELECT 1
    FROM claim_lines AS cl
    WHERE cl.claim_id = c.claim_id
);
"""

correct_df_q3 = pd.read_sql_query(
    correct_sql_q3,
    conn_q3
)

display(correct_df_q3)

correct_total_q3 = correct_df_q3.loc[0, "correct_total"]

print("Correct total:", correct_total_q3)
print("Expected total:", 300)
print("Validation:", "PASS" if correct_total_q3 == 300 else "FAIL")


,correct_total
0,300


Correct total: 300
Expected total: 300
Validation: PASS


In [7]:

# Q3 Step 6: Pre-aggregate detail rows before JOIN

import pandas as pd

preagg_sql_q3 = """
WITH line_summary AS (
    SELECT
        claim_id,
        COUNT(*) AS line_count
    FROM claim_lines
    GROUP BY claim_id
)
SELECT
    c.claim_id,
    c.total_amount,
    COALESCE(ls.line_count, 0) AS line_count
FROM claims AS c
LEFT JOIN line_summary AS ls
    ON c.claim_id = ls.claim_id
ORDER BY c.claim_id;
"""

preagg_df_q3 = pd.read_sql_query(
    preagg_sql_q3,
    conn_q3
)

display(preagg_df_q3)

total_claim_amount_q3 = preagg_df_q3["total_amount"].sum()
total_claim_lines_q3 = preagg_df_q3["line_count"].sum()

print("Total claim amount:", total_claim_amount_q3)
print("Total claim lines:", total_claim_lines_q3)

print(
    "Pre-aggregation validation:",
    "PASS"
    if (
        len(preagg_df_q3) == 2
        and total_claim_amount_q3 == 300
        and total_claim_lines_q3 == 3
    )
    else "FAIL"
)


,claim_id,total_amount,line_count
0,1,100,2
1,2,200,1


Total claim amount: 300
Total claim lines: 3
Pre-aggregation validation: PASS


In [8]:

# Q3 Step 7: Source vs corrected JOIN reconciliation

import pandas as pd

recon_sql_q3 = """
WITH line_summary AS (
    SELECT
        claim_id,
        COUNT(*) AS line_count
    FROM claim_lines
    GROUP BY claim_id
),
corrected_join AS (
    SELECT
        c.claim_id,
        c.total_amount,
        COALESCE(ls.line_count, 0) AS line_count
    FROM claims AS c
    LEFT JOIN line_summary AS ls
        ON c.claim_id = ls.claim_id
),
source_metrics AS (
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT claim_id) AS unique_ids,
        COALESCE(SUM(total_amount), 0) AS amount
    FROM claims
),
target_metrics AS (
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT claim_id) AS unique_ids,
        COALESCE(SUM(total_amount), 0) AS amount
    FROM corrected_join
)
SELECT
    s.row_count AS source_count,
    t.row_count AS target_count,
    s.unique_ids AS source_unique,
    t.unique_ids AS target_unique,
    s.amount AS source_amount,
    t.amount AS target_amount,
    t.row_count - t.unique_ids AS target_duplicates
FROM source_metrics AS s
CROSS JOIN target_metrics AS t;
"""

recon_df_q3 = pd.read_sql_query(
    recon_sql_q3,
    conn_q3
)

display(recon_df_q3)

r = recon_df_q3.iloc[0]

passed_q3 = (
    r["source_count"] == r["target_count"]
    and r["source_unique"] == r["target_unique"]
    and r["source_amount"] == r["target_amount"]
    and r["target_duplicates"] == 0
)

print(
    "Q3 RECONCILIATION:",
    "PASS" if passed_q3 else "FAIL"
)


,source_count,target_count,source_unique,target_unique,source_amount,target_amount,target_duplicates
0,2,2,2,2,300,300,0


Q3 RECONCILIATION: PASS


In [9]:

# Q3 Step 8: Bidirectional EXCEPT reconciliation

import pandas as pd

except_sql_q3 = """
WITH line_summary AS (
    SELECT claim_id, COUNT(*) AS line_count
    FROM claim_lines
    GROUP BY claim_id
),
corrected_join AS (
    SELECT c.claim_id, c.total_amount
    FROM claims c
    LEFT JOIN line_summary ls
        ON c.claim_id = ls.claim_id
),
source_minus_target AS (
    SELECT claim_id, total_amount FROM claims
    EXCEPT
    SELECT claim_id, total_amount FROM corrected_join
),
target_minus_source AS (
    SELECT claim_id, total_amount FROM corrected_join
    EXCEPT
    SELECT claim_id, total_amount FROM claims
)
SELECT
    (SELECT COUNT(*) FROM source_minus_target)
        AS source_minus_target,
    (SELECT COUNT(*) FROM target_minus_source)
        AS target_minus_source;
"""

except_df_q3 = pd.read_sql_query(
    except_sql_q3,
    conn_q3
)

display(except_df_q3)

source_diff = int(except_df_q3.loc[0, "source_minus_target"])
target_diff = int(except_df_q3.loc[0, "target_minus_source"])

print(
    "Record-level reconciliation:",
    "PASS" if source_diff == 0 and target_diff == 0
    else "FAIL"
)


,source_minus_target,target_minus_source
0,0,0


Record-level reconciliation: PASS


In [10]:

# Q3 Step 9: Final validation

source_df_q3 = pd.read_sql_query(
    "SELECT claim_id, total_amount FROM claims",
    conn_q3
)

assert len(source_df_q3) == 2, "Unexpected source row count"

assert source_df_q3["claim_id"].is_unique, (
    "Duplicate claim IDs found"
)

assert source_df_q3["total_amount"].sum() == 300, (
    "Incorrect source claim total"
)

assert total_claim_amount_q3 == 300, (
    "Pre-aggregated JOIN total mismatch"
)

assert total_claim_lines_q3 == 3, (
    "Claim-line count mismatch"
)

assert source_diff == 0 and target_diff == 0, (
    "Record-level reconciliation failed"
)

print("Q3 FINAL VALIDATION: PASS")


Q3 FINAL VALIDATION: PASS


In [11]:

from pathlib import Path

project_q3 = Path.home() / "question_03_join_fanout"

readme_q3 = """# SQL Question 3 — JOIN Fan-Out and Incorrect Aggregation

## Business Problem
An insurance claim report returned a total of 400,
although the correct claim-header total was 300.

## Root Cause
A one-to-many JOIN between claims and claim_lines
repeated claim-level amounts.

## Techniques Practiced
- INNER JOIN and LEFT JOIN
- JOIN cardinality and data grain
- COUNT and GROUP BY
- EXISTS
- Pre-aggregation using CTEs
- COALESCE
- Source-to-target reconciliation
- Bidirectional EXCEPT
- Python assertions

## Results
| Validation | Result |
|---|---|
| Incorrect joined total | 400 |
| Correct claim total | 300 |
| Claim records | 2 |
| Claim-line records | 3 |
| Source minus target | 0 |
| Target minus source | 0 |
| Final validation | PASS |

## Tools
Python, SQLite, pandas, JupyterLab

## Key Learning
Always aggregate financial measures at their
correct business grain. A successful JOIN does
not guarantee correct aggregation.
"""

readme_path_q3 = project_q3 / "README.md"
readme_path_q3.write_text(readme_q3, encoding="utf-8")

print("README created:", readme_path_q3)
print("README exists:", readme_path_q3.is_file())


README created: /home/itv028683/question_03_join_fanout/README.md
README exists: True


In [12]:

# Q3 Step 10B: Verify files for GitHub

from pathlib import Path

project_q3 = Path.home() / "question_03_join_fanout"

print("PROJECT FILES:")
for file in sorted(project_q3.iterdir()):
    print(file.name)

print("\nGITHUB READINESS:")

readme_exists = (project_q3 / "README.md").is_file()
notebook_exists = (
    project_q3 / "SQL_Question_03_Join_Fanout.ipynb"
).is_file()

print("README present:", readme_exists)
print("Notebook present:", notebook_exists)

print(
    "Ready for GitHub:",
    readme_exists and notebook_exists
)


PROJECT FILES:
.ipynb_checkpoints
README.md
SQL_Question_03_Join_Fanout.ipynb.ipynb
claims_join.db

GITHUB READINESS:
README present: True
Notebook present: False
Ready for GitHub: False


In [13]:

# Q3 Step 10C: Fix double .ipynb extension

from pathlib import Path

project_q3 = Path.home() / "question_03_join_fanout"

old_path = project_q3 / "SQL_Question_03_Join_Fanout.ipynb.ipynb"
new_path = project_q3 / "SQL_Question_03_Join_Fanout.ipynb"

if new_path.exists():
    print("Correct filename already exists. No rename performed.")
elif old_path.is_file():
    old_path.rename(new_path)
    print("Notebook renamed successfully!")
else:
    print("Original notebook not found. No rename performed.")

print("README present:", (project_q3 / "README.md").is_file())
print("Notebook present:", new_path.is_file())
print(
    "Ready for GitHub:",
    (project_q3 / "README.md").is_file()
    and new_path.is_file()
)


Notebook renamed successfully!
README present: True
Notebook present: True
Ready for GitHub: True


In [14]:

# Q3 Step 10D: Prepare clean GitHub files

from pathlib import Path
import shutil

source_q3 = Path.home() / "question_03_join_fanout"

github_q3 = Path.home() / "question_03_github_upload"

github_q3.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "README.md",
    "SQL_Question_03_Join_Fanout.ipynb"
]

for filename in files_to_copy:
    source_file = source_q3 / filename
    target_file = github_q3 / filename

    if not source_file.is_file():
        raise FileNotFoundError(source_file)

    shutil.copy2(source_file, target_file)

    print("Copied:", filename)

print("\nFILES READY FOR GITHUB:")

for file in sorted(github_q3.iterdir()):
    if file.is_file():
        print(file.name)

print("\nGitHub preparation: PASS")


Copied: README.md
Copied: SQL_Question_03_Join_Fanout.ipynb

FILES READY FOR GITHUB:
README.md
SQL_Question_03_Join_Fanout.ipynb

GitHub preparation: PASS


In [16]:
# Q3 Step 10E: Check GitHub notebook contents

import json
from pathlib import Path

notebook_path_q3 = (
    Path.home()
    / "question_03_github_upload"
    / "SQL_Question_03_Join_Fanout.ipynb"
)

with notebook_path_q3.open(
    "r", encoding="utf-8"
) as file:
    notebook_q3 = json.load(file)

code_cells_q3 = [
    "".join(cell.get("source", []))
    for cell in notebook_q3.get("cells", [])
    if cell.get("cell_type") == "code"
]

all_code_q3 = "\n".join(code_cells_q3)

checks_q3 = {
    "Creates claims table": (
        "CREATE TABLE IF NOT EXISTS claims" in all_code_q3
    ),
    "Creates claim lines table": (
        "CREATE TABLE IF NOT EXISTS claim_lines" in all_code_q3
    ),
    "Contains sample INSERTs": (
        "INSERT OR IGNORE INTO claims" in all_code_q3
        and "INSERT OR IGNORE INTO claim_lines" in all_code_q3
    ),
    "Contains JOIN practical": (
        "INNER JOIN claim_lines" in all_code_q3
    ),
    "Contains final validation": (
        "Q3 FINAL VALIDATION: PASS" in all_code_q3
    )
}

for check, passed in checks_q3.items():
    print(f"{check}: {'PASS' if passed else 'CHECK'}")

print("\nNotebook code cells:", len(code_cells_q3))

print(
    "Reproducibility content check:",
    "PASS" if all(checks_q3.values()) else "REVIEW REQUIRED"
)

Creates claims table: PASS
Creates claim lines table: PASS
Contains sample INSERTs: PASS
Contains JOIN practical: PASS
Contains final validation: PASS

Notebook code cells: 12
Reproducibility content check: PASS
